# 🧠 AccessLens — AI Accessibility Audit Tool
## End-to-End Training & Deployment Notebook

**Steps:**
1. Upload project files
2. Install dependencies
3. Generate synthetic dataset (500 samples)
4. Train EfficientNet-B0 CNN model
5. Start web server + public URL

**Runtime:** Set to **T4 GPU** → Runtime → Change runtime type → T4 GPU

## Step 1: Upload Project
Upload your `accesslens.zip` containing `backend/` and `frontend/` folders.

In [ ]:
from google.colab import files
import zipfile, os

print('📁 Upload your accesslens.zip file...')
uploaded = files.upload()

for fname in uploaded:
    print(f'Extracting {fname}...')
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/accesslens')

os.chdir('/content/accesslens')
!ls -la
print('\n✅ Project uploaded!')

## Step 2: Install Dependencies

In [ ]:
%%time
# Install system deps
print('📦 Installing system dependencies...')
!sudo apt-get install -y -qq zstd chromium-browser chromium-chromedriver > /dev/null 2>&1

# Install Python packages
print('📦 Installing Python packages...')
!pip install -q fastapi uvicorn beautifulsoup4 selenium webdriver-manager httpx pydantic python-multipart lxml Pillow cssutils torch torchvision scikit-learn matplotlib tqdm pyngrok

print('\n✅ All dependencies installed!')

## Step 3: Generate Synthetic Dataset
Creates 500 labeled accessibility violation screenshots.

In [ ]:
%%time
import sys, os, logging
os.chdir('/content/accesslens')
sys.path.insert(0, '/content/accesslens')
logging.basicConfig(level=logging.INFO, format='%(message)s')

from backend.ml.dataset_generator import DatasetGenerator

print('🎨 Generating 500 synthetic accessibility screenshots...')
print('This creates HTML pages with deliberate WCAG violations,')
print('renders them, and saves labeled images.\n')

gen = DatasetGenerator(output_dir='dataset', num_samples=500)
gen.generate()

print(f'\n✅ Dataset generated: {len(gen.metadata)} samples')
print(f'   Images: dataset/images/')
print(f'   Metadata: dataset/metadata.json')
print(f'   Splits: dataset/splits.json')

In [ ]:
# Verify dataset
import json
from PIL import Image
import matplotlib.pyplot as plt

with open('dataset/metadata.json') as f:
    meta = json.load(f)

print(f'Total samples: {meta["num_samples"]}')
print(f'Classes: {meta["class_names"]}')

# Show class distribution
from collections import Counter
label_counts = Counter()
for s in meta['samples']:
    for name in s['label_names']:
        label_counts[name] += 1

print('\nViolation distribution:')
for name, count in sorted(label_counts.items()):
    print(f'  {name}: {count}')

# Show sample images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(meta['samples']):
        img = Image.open(f'dataset/{meta["samples"][i]["image"]}')
        ax.imshow(img)
        labels = meta['samples'][i]['label_names'] or ['clean']
        ax.set_title(', '.join(labels), fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Dataset Images', fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Train the Model
Fine-tunes EfficientNet-B0 on the synthetic dataset.

In [ ]:
%%time
import torch
from backend.ml.train import train_model
from backend.ml.model import count_parameters, get_model

# Print model info
model = get_model(pretrained=True)
params = count_parameters(model)
print(f'Model: AccessibilityNet (EfficientNet-B0)')
print(f'Parameters: {params["total_millions"]}')
print(f'Trainable: {params["trainable"]:,}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print()

# Train
history = train_model(
    dataset_dir='dataset',
    epochs=25,
    batch_size=32,
    learning_rate=1e-4,
    freeze_backbone=False,
)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt
import json

with open('dataset/training_history.json') as f:
    h = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(h['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(h['val_loss'], label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# F1
ax2.plot(h['train_f1'], label='Train F1', linewidth=2)
ax2.plot(h['val_f1'], label='Val F1', linewidth=2)
ax2.axhline(y=h['best_val_f1'], color='r', linestyle='--', alpha=0.5, label=f'Best: {h["best_val_f1"]:.3f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Macro F1')
ax2.set_title('Training & Validation F1 Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle(f'AccessibilityNet Training (Best Epoch: {h["best_epoch"]})', fontsize=14)
plt.tight_layout()
plt.show()

# Per-class F1
print('\nPer-class F1 (best epoch):')
best = h['per_class_f1'][h['best_epoch']-1]
for cls, f1 in best.items():
    bar = '█' * int(f1 * 30)
    print(f'  {cls:20s} {f1:.3f} {bar}')

## Step 5: Test the Model

In [ ]:
from backend.ml.inference import AccessibilityInference
from PIL import Image
import glob

inf = AccessibilityInference(model_path='dataset/accessibility_model.pth')
print(f'Model loaded: {inf.is_available}')

# Test on random samples
imgs = sorted(glob.glob('dataset/images/*.png'))[:5]
for img_path in imgs:
    results = inf.predict(img_path)
    probs = inf.get_all_probabilities(img_path)
    print(f'\n{img_path.split("/")[-1]}:')
    print(f'  Detections: {[r["category"] for r in results] or "None (clean)"}')
    print(f'  Probabilities: { {k: f"{v:.2f}" for k, v in probs.items()} }')

## Step 6: Start Web Server

In [ ]:
# Start FastAPI server in background
import subprocess, time

proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/accesslens'
)
time.sleep(3)
print('✅ Server started on port 8000')

In [ ]:
# Create public URL with ngrok
from pyngrok import ngrok

public_url = ngrok.connect(8000)
print(f'\n🌐 Your AccessLens is live at:')
print(f'   {public_url.public_url}')
print(f'\nOpen this URL in your browser to use the tool!')